# L5B15 Decision Data — Ingestion

Parses raw Pega decision JSON exports, extracts L5B15 scoring events, and writes a clean Parquet file to `data/processed/`.

**To add new data:** drop additional `data*.json` files into `data/` and re-run this notebook.
Deduplication on `modelExecutionID` ensures records are never double-counted.

In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import polars as pl
from IPython.display import HTML, display
from tabulate import tabulate
from tqdm import tqdm

from my_project.parsing import extract_l5b15_rows, parse_common_inputs

print("Imports OK")

In [ ]:
DATA_DIR = Path("../data")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROCESSED_DIR / "l5b15_decisions.parquet"

# All matching raw JSON files — add new files here or to data/ and re-run
L5B15_FILES = sorted(DATA_DIR.glob("data*.json"))
print(f"Found {len(L5B15_FILES)} source file(s):")
for f in L5B15_FILES:
    print(f"  {f.name}")

## Load & parse raw records

In [ ]:
all_rows = []

for filepath in L5B15_FILES:
    print(f"\nProcessing {filepath.name}...")
    with open(filepath, encoding="utf-8") as f:
        total = sum(1 for _ in f)

    records = []
    with open(filepath, encoding="utf-8") as f:
        for line in tqdm(f, total=total, desc=filepath.name):
            records.append(json.loads(line))

    df_raw = pl.DataFrame(records)
    print(f"  {df_raw.shape[0]:,} raw records, {df_raw.shape[1]} columns")

    rows = extract_l5b15_rows(df_raw)
    print(f"  {len(rows):,} L5B15 scoring events extracted")
    all_rows.extend(rows)

print(f"\nTotal rows before deduplication: {len(all_rows):,}")

In [ ]:
df_model = pd.DataFrame(all_rows)

# Deduplicate on modelExecutionID (safe to re-run with overlapping exports)
before = len(df_model)
df_model = df_model.drop_duplicates(subset=["modelExecutionID"])
print(f"Rows after dedup: {len(df_model):,}  (removed {before - len(df_model):,} duplicates)")
print(f"Columns: {df_model.shape[1]}")
df_model.head(3)

## Partition overview

## Partition overview

Which partition keys exist and how many distinct L5B15 scoring events per partition name?

In [ ]:
from tabulate import tabulate
from IPython.display import HTML, display
from collections import Counter

name_counts = Counter(df_model["pyName"].dropna())
table_data = [[rank + 1, name, count] for rank, (name, count) in enumerate(name_counts.most_common(20))]
table_str = tabulate(table_data, headers=["#", "Partition Name", "Count"], tablefmt="grid")
display(HTML(f"<pre>{table_str}</pre>"))

Over the 3-day observation window, **14,895 L5B15 scoring events** were extracted across all model partitions. This is the decision-level dataset used for data quality assessment and surrogate modelling.

## 6. Data Quality Assessment

Before building a surrogate model we assess the quality of the extracted feature matrix: which feature namespaces are present, how much data is missing, which columns are constant, and how features correlate with the propensity target.

## Save to Parquet

In [ ]:
df_model.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved {len(df_model):,} rows → {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")